In [27]:
!pip3 install pyspark~=4.0.0

In [28]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as func
import pyspark.sql.types as type
from math import sqrt

In [29]:
spark = SparkSession.builder.getOrCreate()
spark

# 1. Найти велосипед с максимальным временем пробега.

In [30]:
trips = spark.read.format('csv').option('header', 'true').load("trip.csv")

In [31]:
# Группируем по bike_id и вычисляем максимальную длительность поездки
max_duration_per_bike = (
    trips.groupBy('bike_id').agg(func.max(func.col("duration").cast(type.IntegerType())).alias("duration"))
)

# Сортируем по убыванию
top_longest_trip = max_duration_per_bike.orderBy(func.col('duration').desc())
top_longest_trip.show(1)

+-------+--------+
|bike_id|duration|
+-------+--------+
|    535|17270400|
+-------+--------+
only showing top 1 row


# 2. Найти наибольшее геодезическое расстояние между станциями.

In [32]:
stations = spark.read.format('csv').option('header', 'true').load("station.csv")

In [33]:
# Выбираем столбцы 'id', 'lat' и 'long'
stations_data = stations.select("id", "lat", "long")
stations_data.show(5)

# Создаем комбинации станций и выбираем только различные комбинации
combo = stations_data.selectExpr('id as A', 'lat as A_lat', 'long as A_long').join(
    stations_data.selectExpr('id as B', 'lat as B_lat', 'long as B_long'))
dif_combo = combo[combo.A != combo.B]

def distance(ax, ay, bx, by):
    return sqrt((ax - bx) ** 2 + (ay - by) ** 2)

dists = dif_combo.rdd.map(
    lambda row: (row.A, row.B, distance(float(row.A_lat), float(row.A_long), float(row.B_lat), float(row.B_long)))
)

# Находим максимум
stations_data = dists.max(lambda row: row[2])
stations_data


+---+------------------+-------------------+
| id|               lat|               long|
+---+------------------+-------------------+
|  2|         37.329732|-121.90178200000001|
|  3|         37.330698|        -121.888979|
|  4|         37.333988|        -121.894902|
|  5|         37.331415|          -121.8932|
|  6|37.336721000000004|        -121.894074|
+---+------------------+-------------------+
only showing top 5 rows


('16', '60', 0.7058482821754397)

# 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [34]:
# Выбираем столбцы 'id', 'bike_id', 'start_station_id' и 'end_station_id', фильтруем по 'bike_id'= 535 и сортируем по 'id'
filtered_trips = (
    trips.select("id", "bike_id", "start_station_id", "end_station_id").filter(func.col("bike_id") == 535)
    .orderBy(func.col("id").cast(type.IntegerType()))
)

filtered_trips.show(filtered_trips.count())

+------+-------+----------------+--------------+
|    id|bike_id|start_station_id|end_station_id|
+------+-------+----------------+--------------+
|  4966|    535|              47|            70|
|  5067|    535|              70|            69|
|  5179|    535|              69|            77|
|  5199|    535|              77|            64|
|  7806|    535|              61|            42|
| 11422|    535|              58|            72|
| 12245|    535|              72|            47|
| 12485|    535|              47|            60|
| 12558|    535|              60|            46|
| 13107|    535|              46|            77|
| 13423|    535|              77|            77|
| 14380|    535|              77|            62|
| 14581|    535|              62|            61|
| 15231|    535|              55|            61|
| 15242|    535|              61|            60|
| 15347|    535|              60|            41|
| 15605|    535|              41|            50|
| 15611|    535|    

# 4. Найти количество велосипедов в системе.

In [35]:
count_bikes = max_duration_per_bike.count()

count_bikes

700

# 5. Найти пользователей потративших на поездки более 3 часов.

In [36]:
# Группируем по столбцу 'zip_code' и вычисляем максимальную продолжительность поездки
output_filtered = (
    trips.groupBy('zip_code').agg(func.max(func.col("duration").cast(type.IntegerType())).alias("duration"))
)

# Оставляем записи, где продолжительность поездки больше или равна 3х часов
output_filtered = output_filtered.filter(func.col("duration") >= 3*60*60)

output_filtered.show()

+--------+--------+
|zip_code|duration|
+--------+--------+
|   94102|  464952|
|   95134|   82487|
|   84606|   14575|
|   80305|   74749|
|   60070|   26540|
|   91910|   20243|
|    2136|   16010|
|   11722|   12173|
|   94610|   76287|
|   94404|   63504|
|   80301|   36931|
|   94309|   18484|
|   97239|  193241|
|   94592|   26999|
|    7650|   20150|
|   92374|   17156|
|   11106|   13773|
|   93013|   25116|
|   30324|   17117|
|   16303|   13072|
+--------+--------+
only showing top 20 rows
